# GraphSAGE Comparison — Google Colab (T4 GPU)

Run these cells **in order**, top to bottom.

**First:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.

**What this notebook does:**

This is the same official GIN codebase as before, but run with the settings the GIN paper itself
uses to *approximate GraphSAGE* inside its own architecture (max-pooling of neighbors + a plain
1-layer transform, instead of GIN's sum-pooling + multi-layer MLP) — quoting the paper: *"max–1-layer
correspond[s] to GraphSAGE, up to minor architecture modifications."* Same training protocol,
same 10-fold CV, same everything else — only the aggregation changes. No new code, no new
dependencies, same tested pipeline as the GIN run.

1. Clones the official GIN repo (same source as before)
2. Runs the GraphSAGE-approximation settings on **IMDB-BINARY** and **IMDB-MULTI** (paper's original data)
3. After you upload two files, runs the same settings on **our new scraped data** — `IMDBBINARY-NEW` and `IMDBMULTI-NEW`
4. Prints one final summary block — **copy that block back into the chat when it's done**

Each dataset run is 10-fold cross-validation × 350 epochs, same as the GIN runs. Results save after
every fold, so a Colab disconnect partway through doesn't lose finished work.

**Expected ballpark, from the paper's own Table 1 (GraphSAGE / Max–1-Layer row):**
IMDB-BINARY ≈ 72.3 ± 5.3%, IMDB-MULTI ≈ 50.9 ± 2.2% — a bit below GIN's 75.1% / 52.3% on the same
original data. The interesting question is whether that gap holds, shrinks, or vanishes on our new,
denser scraped data.


## 0. Check the GPU is actually connected

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to: Runtime > Change runtime type > select GPU (T4) > Save, "
          "then Runtime > Restart session, then re-run this cell.")


## 1. Clone the official GIN repo and install dependencies

In [ ]:
!git clone -q https://github.com/weihua916/powerful-gnns.git
%cd powerful-gnns
!unzip -q -o dataset.zip
!pip install -q networkx tqdm
print("Setup done. Datasets available:", end=" ")
!ls dataset


## 1b. Fix a device-placement bug in the official code (GPU-only slowdown)

The official repo has a real bug that only shows up on GPU: its max-pooling neighbor-aggregation
path (`models/graphcnn.py`) builds a lookup tensor called `padded_neighbor_list` but never moves it
to the GPU — unlike the sum/average-pooling path a few lines below it, which does this correctly
(`Adj_block.to(self.device)`). On GPU this means every single training iteration indexes a CUDA
tensor with a CPU tensor, forcing a slow implicit CPU↔GPU sync each time. It's invisible on CPU
(no device mismatch there), which is why local CPU testing looked fine but a full GPU run stalls.

Since GraphSAGE's approximation in this codebase specifically uses max-pooling, this bug hits
every run in this notebook. The fix below is a single line, mirrors exactly what the working
sum/average path already does, and does not change any computation — same math, just correctly
placed on the GPU.


In [ ]:
path = "models/graphcnn.py"
src = open(path).read()
old = "        return torch.LongTensor(padded_neighbor_list)"
new = "        return torch.LongTensor(padded_neighbor_list).to(self.device)"
assert old in src, "expected line not found - official repo may have changed, needs a manual look"
src = src.replace(old, new)
open(path, "w").write(src)
print("Patched models/graphcnn.py:")
print("  before:", old.strip())
print("  after: ", new.strip())


## 2. Upload our scraped datasets

Run the next cell, then in the file picker select **both** of these files from your computer
(from `new_data/processed/.../` in the project folder):

- `IMDBBINARY-NEW.txt`
- `IMDBMULTI-NEW.txt`

(You do **not** need the `_metadata.json` files for this — just the two `.txt` files.)


In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()

for fname in uploaded.keys():
    if not fname.endswith(".txt"):
        print(f"skipping {fname} (not a .txt dataset file)")
        continue
    dsname = fname[:-4]
    os.makedirs(f"dataset/{dsname}", exist_ok=True)
    shutil.move(fname, f"dataset/{dsname}/{fname}")
    print(f"placed dataset/{dsname}/{fname}")


## 3. Training / cross-validation driver

Same protocol as the GIN notebook: run all 10 folds, average the 10 test-accuracy curves
epoch-by-epoch, pick the single best epoch, report mean ± std across folds at that epoch.

The only difference from the GIN run is which flags get passed to `main.py` — this driver takes
a `--tag` so results are saved as `<dataset>_<tag>_cv_curves.json`, keeping them clearly separate
from any GIN results you might combine into the same results folder later.


In [ ]:
%%writefile run_variant.py
import argparse
import json
import re
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("results")
EPOCH_RE = re.compile(r"loss training: ([\d.]+)\naccuracy train: ([\d.]+) test: ([\d.]+)")


def run_fold(dataset, fold_idx, extra_args, epochs):
    cmd = [sys.executable, "main.py", "--dataset", dataset,
           "--fold_idx", str(fold_idx), "--epochs", str(epochs)] + extra_args
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"fold {fold_idx} failed:\n{proc.stderr[-3000:]}")
    matches = EPOCH_RE.findall(proc.stdout)
    if len(matches) != epochs:
        raise RuntimeError(
            f"fold {fold_idx}: expected {epochs} epochs, parsed {len(matches)}\n{proc.stdout[-1000:]}"
        )
    return [{"loss": float(l), "train_acc": float(tr), "test_acc": float(te)} for l, tr, te in matches]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--tag", default="run")
    ap.add_argument("--epochs", type=int, default=350)
    ap.add_argument("--extra", nargs=argparse.REMAINDER, default=[])
    args = ap.parse_args()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = RESULTS_DIR / f"{args.dataset}_{args.tag}_cv_curves.json"

    all_folds = []
    t0 = time.time()
    for fold_idx in range(10):
        fstart = time.time()
        curve = run_fold(args.dataset, fold_idx, args.extra, args.epochs)
        all_folds.append(curve)
        print(f"[{args.dataset}/{args.tag}] fold {fold_idx} done in {time.time() - fstart:.1f}s "
              f"(test_acc={curve[-1]['test_acc']:.4f})", flush=True)
        out_path.write_text(json.dumps(
            {"dataset": args.dataset, "tag": args.tag, "epochs": args.epochs,
             "extra_args": args.extra, "folds": all_folds},
            indent=2))

    n_epochs = args.epochs
    avg_curve = [sum(fold[e]["test_acc"] for fold in all_folds) / len(all_folds) for e in range(n_epochs)]
    best_epoch = max(range(n_epochs), key=lambda e: avg_curve[e])
    accs = [fold[best_epoch]["test_acc"] for fold in all_folds]
    mean_acc = sum(accs) / len(accs)
    std_acc = (sum((a - mean_acc) ** 2 for a in accs) / len(accs)) ** 0.5

    summary = {
        "dataset": args.dataset, "tag": args.tag, "epochs": n_epochs, "extra_args": args.extra,
        "best_epoch_0indexed": best_epoch, "mean_test_acc_at_best_epoch": mean_acc,
        "std_test_acc_at_best_epoch": std_acc, "per_fold_acc_at_best_epoch": accs,
        "total_runtime_sec": time.time() - t0,
    }
    out_path.write_text(json.dumps(
        {"dataset": args.dataset, "tag": args.tag, "epochs": args.epochs, "extra_args": args.extra,
         "folds": all_folds, "summary": summary}, indent=2))
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


## 4. Run: IMDB-BINARY, GraphSAGE settings (paper's original data)

In [ ]:
!python run_variant.py --dataset IMDBBINARY --tag graphsage --epochs 350 --extra --degree_as_tag --neighbor_pooling_type max --num_mlp_layers 1

## 5. Run: IMDB-MULTI, GraphSAGE settings (paper's original data)

In [ ]:
!python run_variant.py --dataset IMDBMULTI --tag graphsage --epochs 350 --extra --degree_as_tag --neighbor_pooling_type max --num_mlp_layers 1

## 6. Run: IMDBBINARY-NEW, GraphSAGE settings (our scraped data)

In [ ]:
!python run_variant.py --dataset IMDBBINARY-NEW --tag graphsage --epochs 350 --extra --degree_as_tag --neighbor_pooling_type max --num_mlp_layers 1

## 7. Run: IMDBMULTI-NEW, GraphSAGE settings (our scraped data)

In [ ]:
!python run_variant.py --dataset IMDBMULTI-NEW --tag graphsage --epochs 350 --extra --degree_as_tag --neighbor_pooling_type max --num_mlp_layers 1

## 8. Final summary — copy this whole block back into the chat

In [ ]:
import json
import glob

print("=" * 60)
print("FINAL SUMMARY (GraphSAGE settings)")
print("=" * 60)
for f in sorted(glob.glob("results/*_cv_curves.json")):
    d = json.load(open(f))
    s = d.get("summary")
    if s:
        print(f"{s['dataset']:20s} [{s['tag']:10s}] mean={s['mean_test_acc_at_best_epoch']*100:5.1f}%  "
              f"std={s['std_test_acc_at_best_epoch']*100:4.1f}%  "
              f"best_epoch={s['best_epoch_0indexed']+1:3d}  "
              f"runtime={s['total_runtime_sec']/60:5.1f} min")
    else:
        print(f"{f}: no summary yet (run incomplete)")


## 9. (Optional) Download raw results to bring back

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("graphsage_results", "zip", "results")
files.download("graphsage_results.zip")
